# pyRockImageLab — Example: core box photo to high-resolution mosaic

A small, anonymized example (`example_data/ExampleCore/`) that reproduces the 2563–2566 ft high-resolution (HR) core mosaic.

- `ExampleCore_00_box.jpg`: overview photo of the core box (three 3-ft segments)
- `ExampleCore_01..06_detail.jpg`: close-up photos of segment 1 (2563–2566 ft)
- `_picks.json`: saved corner picks, so the notebook runs without clicking

Workflow: crop and ortho-warp each detail photo → stitch them (SIFT + RANSAC) → ortho-warp the mosaic → save PNG.
Outputs go to `outputs/` as PNG only.

In [1]:
# %pip install --quiet pyrockimagelab pillow-heif opencv-python matplotlib PyQt5

In [2]:
from pathlib import Path
import matplotlib
import pyrockimagelab as pril
from pyrockimagelab import (QuadPicker, list_source_images, load_any_image, load_picks,
                            ortho_warp_quad, stitch_chain, release_working_memory)
from pyrockimagelab.image_processing.warp import _apply_hdr_enhancement, clamp_quad_to_image, trim_empty_rows
import numpy as np
from PIL import Image

INTERACTIVE = False          # True: open the Qt pickers to (re)pick corners
if not INTERACTIVE:
    matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt

FOLDER = Path("example_data/ExampleCore/2563-2572")
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)      # PNG outputs (not tracked by git)
FIG = Path("figures"); FIG.mkdir(exist_ok=True)       # small README figure
HR_SAVE_SCALE = 0.5   # 1.0 = full resolution (~50 MB PNG); 0.5 keeps it ~13 MB
picks = load_picks(FOLDER)
sources = list_source_images(FOLDER)
print(f"pyrockimagelab {pril.__version__}")
for i, p in enumerate(sources):
    print(f"[{i}] {p.name}")

pyrockimagelab 0.0.3
[0] ExampleCore_00_box.jpg
[1] ExampleCore_01_detail.jpg
[2] ExampleCore_02_detail.jpg
[3] ExampleCore_03_detail.jpg
[4] ExampleCore_04_detail.jpg
[5] ExampleCore_05_detail.jpg
[6] ExampleCore_06_detail.jpg


## Step 1 — Box photo and segment 1 (low resolution)

The saved quad marks segment 1 on the box photo. The figure below is the same corner-picking window you get interactively (TL/TR/BR/BL markers).

In [3]:
box = load_any_image(FOLDER / picks["box_photo"])
quad = picks["box_quads"][f'{picks["box_photo"]}::seg1']

picker = QuadPicker(box, initial_quad=quad, title="LR Segment 1/3 — drag corners, Enter to confirm")
if INTERACTIVE:
    quad = picker.run()          # opens the Qt window

lr = _apply_hdr_enhancement(ortho_warp_quad(box, quad))
Image.fromarray(lr).save(OUT / "ExampleCore_2563-2566.png", optimize=True)
print("LR segment:", lr.shape[1], "x", lr.shape[0], "px")

# Figure A: crop selection (left) and the cropped box segment (right), same height
picker.fig.canvas.draw()
ui = np.asarray(picker.fig.canvas.buffer_rgba())[..., :3]
plt.close(picker.fig)
rows, cols = np.where(ui.min(axis=2) < 250)
ui = ui[rows.min():rows.max() + 1, cols.min():cols.max() + 1]          # trim white margins
H = ui.shape[0]
seg = np.asarray(Image.fromarray(lr).resize((round(lr.shape[1] * H / lr.shape[0]), H), Image.LANCZOS))
gap = np.full((H, 30, 3), 255, np.uint8)
Image.fromarray(np.hstack([ui, gap, seg])).save(FIG / "ExampleCore_2563-2566_box_crop.png", optimize=True)

LR segment: 305 x 3610 px


## Step 2 — Crop each detail photo with its saved quad

In [4]:
names = [sources[i].name for i in picks["hr_assignments"]["seg1"]]
crops = []
for n in names:
    img = load_any_image(FOLDER / n)
    q = picks["hr_photo_quads"][n]
    if INTERACTIVE:
        q = QuadPicker(img, initial_quad=q, title=n).run() or q
    crops.append(_apply_hdr_enhancement(ortho_warp_quad(img, q)))
    print(f"{n}: {crops[-1].shape[1]} x {crops[-1].shape[0]} px")

ExampleCore_01_detail.jpg: 1627 x 3777 px


ExampleCore_02_detail.jpg: 1588 x 4011 px


ExampleCore_03_detail.jpg: 1582 x 4020 px


ExampleCore_04_detail.jpg: 1603 x 4017 px


ExampleCore_05_detail.jpg: 1633 x 4017 px


ExampleCore_06_detail.jpg: 1708 x 3468 px


## Step 3 — Stitch, ortho-correct and save the HR mosaic

The saved mosaic quad is clipped to the stitched mosaic and empty rows are trimmed, so the mosaic has no black margin and spans the same core interval as the box-photo segment.

In [5]:
mosaic = stitch_chain(1, crops)
quad = clamp_quad_to_image(picks["hr_mosaic_quad_seg1"], mosaic)
hr = trim_empty_rows(ortho_warp_quad(mosaic, quad))
print("HR mosaic:", hr.shape[1], "x", hr.shape[0], "px")
h, w = hr.shape[:2]
Image.fromarray(hr).resize((round(w * HR_SAVE_SCALE), round(h * HR_SAVE_SCALE)), Image.LANCZOS) \
    .save(OUT / "ExampleCore_2563-2566_hr.png", optimize=True)

# Figure B: cropped close-ups that were stitched (left) beside the stitched HR mosaic (right)
n = len(crops)
fig = plt.figure(figsize=(4.5, 9))
gs = fig.add_gridspec(n, 2, hspace=0.08, wspace=0.12)
for i, c in enumerate(crops):
    a = fig.add_subplot(gs[i, 0]); a.imshow(c, aspect="auto"); a.axis("off")
    a.text(0.03, 0.93, f"{i + 1}", transform=a.transAxes, fontsize=8, color="yellow", va="top", weight="bold")
    if i == 0: a.set_title(f"{n} cropped close-ups\n(stitched in order 1→{n})", fontsize=9)
a = fig.add_subplot(gs[:, 1]); a.imshow(hr, aspect="auto"); a.axis("off")
a.set_title("Stitched HR mosaic\n2563–2566 ft", fontsize=9)
fig.savefig(FIG / "ExampleCore_2563-2566_hr.png", dpi=110, bbox_inches="tight")
plt.close(fig)
del mosaic, crops
release_working_memory()

  pair 0->1:


    25001+25000 desc, 25001 raw, 294 good
      RANSAC: 180/294 inliers, persp=1.20e-05
    -> accepted (180 inliers)
  pair 1->2:


    25000+25001 desc, 25000 raw, 78 good
      RANSAC: 39/78 inliers, persp=1.11e-05
    -> accepted (39 inliers)
  pair 2->3:


    25001+25000 desc, 25001 raw, 441 good
      RANSAC: 418/441 inliers, persp=9.02e-06
    -> accepted (418 inliers)
  pair 3->4:


    25000+25001 desc, 25000 raw, 80 good
      RANSAC: 47/80 inliers, persp=1.05e-05
    -> accepted (47 inliers)
  pair 4->5:


    25001+25001 desc, 25001 raw, 54 good
      RANSAC: 22/54 inliers, persp=2.17e-05
    -> accepted (22 inliers)
  Blending 6 images...


HR mosaic: 1571 x 15935 px


  Memory cleared.


### Full interactive run

To pick everything yourself (box photo, segments, HR photo assignment, crops, mosaic) with the package's Qt windows, use `%matplotlib qt` and:

```python
pril.process_folder(folder=FOLDER, uwi="ExampleCore", folder_idx=1, total_folders=1)
```

Note that `process_folder` writes TIFF + PNG next to the depth folder.